# OpenAI Agents SDK Foundations: Agents, Tools, Handoffs, and Guardrails

## 📚 Learning Objectives

In this notebook, you will learn how to:
- **Explain what the OpenAI Agents SDK (`openai-agents`) is** and how it differs from calling the plain `openai` chat/completions client directly
- **Build a minimal `Agent`** with a `name` and `instructions`, and run it with `Runner`
- **Give an agent a function tool** via the `@function_tool` decorator and observe it being called
- **Wire up handoffs** between a triage agent and specialized agents via `handoffs=[...]`, and see which agent actually produced the final answer
- **Implement an input guardrail** with `@input_guardrail` that validates a request before the agent runs, blocking disallowed input via a tripwire

## 🎯 Where This Fits

This notebook lives in `06_Agent_SDKs_First_Party/OpenAI_Agents_SDK/01_Foundations/` — the first-party, framework-native track for OpenAI's own agent-building SDK, distinct from LangGraph/LangChain (Phases 2-4) and from CrewAI/AutoGen/DSPy (Phase 10). It uses the **`openai-agents` SDK's own native configuration** (`Agent`, `Runner`, `function_tool`, `handoff`, `input_guardrail`), not this repo's `helpers.get_llm()` factory — that factory is reserved for LangGraph-phase notebooks, and the OpenAI Agents SDK is a different first-party runtime with its own agent loop.

## 🔑 Key Concepts

- **`Agent`**: A configuration object — `name`, `instructions` (system prompt), optionally `tools`, `handoffs`, `input_guardrails`/`output_guardrails`, and `model`. Building one does not call any API; nothing runs until it is handed to a `Runner`.
- **`Runner`**: The execution harness. `Runner.run(agent, input)` is the async entry point (there is also a blocking `Runner.run_sync(...)`); it drives the agent loop — including any tool calls and any handoffs — until a final answer is produced, returning a `RunResult`.
- **Function tool**: A plain Python function decorated with `@function_tool`. The SDK turns its signature, type hints, and docstring into a callable tool schema automatically, the same "auto-wrap a function" idea you have already seen with ADK's plain-function tools and LangChain's `@tool`.
- **Handoff**: A way for one agent to transfer an entire conversation to another, more specialized agent. A triage `Agent` is configured with `handoffs=[agent_a, agent_b, ...]`; the SDK exposes each handoff to the model as a special tool call, and `RunResult.last_agent` tells you which agent actually produced the final output.
- **Guardrail**: A function decorated with `@input_guardrail` (or `@output_guardrail`) that runs alongside/before the agent and can trip a **tripwire** — raising `InputGuardrailTripwireTriggered` — to block a request before the main agent ever answers it.

## 🚀 Let's Get Started!


## 1. What is the OpenAI Agents SDK, and how is it different from calling `openai` directly?

### What we are going to do

Before writing any code, it is worth being precise about what `openai-agents` actually adds on top of the plain `openai` Python client.

- The plain **`openai` SDK** gives you direct access to the Chat Completions / Responses APIs: you send messages, you get a completion back, and *you* are responsible for writing the loop that inspects the response, executes any tool calls, feeds the tool results back in, decides whether another turn is needed, and decides which "agent" (i.e. which system prompt + tool set) should be handling the conversation at any given moment.
- The **OpenAI Agents SDK** (`openai-agents`, imported as `agents`) is a small, production-oriented framework built *on top of* that same API surface. It packages the agent loop, tool execution, multi-agent handoffs, and input/output validation ("guardrails") into a few composable primitives: `Agent`, `Runner`, `function_tool`, `handoff`, and `input_guardrail`/`output_guardrail`.

**In one sentence:** the plain `openai` client answers *"give me one completion"*; the Agents SDK answers *"run a full agent — possibly handing off to other agents, possibly calling tools, possibly rejecting bad input — until it has a final answer."* That is the same problem space LangGraph solves generically in this repo's Phase 3/5 content, but here it is OpenAI's own, opinionated, first-party answer — we will return to that comparison in the closing section.


In [ ]:
# ================================================================================
# SETUP: Install openai-agents
# ================================================================================
# openai-agents is OpenAI's official Agents SDK. It bundles:
#   - agents.Agent    -> the agent configuration object (name, instructions, tools, ...)
#   - agents.Runner   -> the execution harness that drives the agent loop
#   - agents.function_tool     -> decorator that turns a plain function into a tool
#   - agents.handoff / the `handoffs=[...]` Agent field -> multi-agent handoffs
#   - agents.input_guardrail / agents.output_guardrail -> request/response validation
# It depends on (and re-exports parts of) the plain `openai` client under the hood.
# ================================================================================

!pip install -q openai-agents

In [ ]:
# ================================================================================
# SETUP: Imports and environment
# ================================================================================
# OPENAI_API_KEY is already a documented required env var for this repo (see
# CLAUDE.md). The Agents SDK picks it up automatically from the environment, so
# no explicit client construction is required for the basic flows in this
# notebook.
# ================================================================================

import os

from dotenv import load_dotenv

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), (
    "Set OPENAI_API_KEY in your .env file — the Agents SDK reads it "
    "automatically from the environment."
)

## 2. Agents: The Basic Building Block

### What we are going to do

We will build the smallest useful agent: an `Agent` configured with just a `name` and `instructions` (the system prompt), then run it on a simple query using `Runner.run(...)`.

A few real API details worth calling out (verified against the installed `openai-agents==0.22.0` package rather than assumed):

- `Agent(name: str, instructions: str | Callable | None = None, tools=[...], handoffs=[...], model: str | None = None, input_guardrails=[...], output_guardrails=[...], ...)` — `model` can be left unset, in which case the SDK falls back to its own default model.
- `Runner.run(starting_agent, input, ...)` is a **coroutine** — it must be awaited (Jupyter supports top-level `await`). A blocking `Runner.run_sync(...)` also exists for plain scripts.
- The return value is a `RunResult` dataclass; the field we care about first is `final_output` (the agent's final text/answer).


In [ ]:
# ================================================================================
# STEP 1: Define a minimal Agent
# ================================================================================
# Constructing an Agent does NOT call any API -- it just builds a configuration
# object, exactly like ADK's Agent or CrewAI's Agent in the sibling notebooks.
# Nothing runs until we hand it to Runner.run(...).
# ================================================================================

from agents import Agent, Runner

concierge_agent = Agent(
    name="Concierge Agent",
    instructions=(
        "You are a friendly, concise product concierge. Answer the user's "
        "question in two sentences or fewer."
    ),
)

print(f"Created agent: {concierge_agent.name!r}")

In [ ]:
# ================================================================================
# STEP 2: Run the agent with Runner.run(...)
# ================================================================================
# Runner.run is async, so we await it directly at the top level of the notebook.
# The RunResult it returns carries the final answer on `.final_output`.
# ================================================================================

result = await Runner.run(concierge_agent, "In one sentence, what is a vector database?")

print("--- Final output ---")
print(result.final_output)

### Discussion of the Output

`Runner.run(...)` drove a full turn of the agent loop for us: it sent `concierge_agent`'s instructions plus our input to the model, got back a response with no tool calls or handoffs to process, and returned a `RunResult` whose `final_output` is the plain-text answer. This mirrors the same "build a config object, then hand it to a runner" separation you have already seen with ADK's `Agent` + `Runner` and CrewAI's `Agent` + `Crew.kickoff()` — the object itself is inert until executed.


## 3. Tools: Giving the Agent a Function to Call

### What we are going to do

We will define a plain Python function, decorate it with `@function_tool`, and attach it to a new agent via the `tools=[...]` field. Just like ADK's plain-function tools and LangChain's `@tool`, the SDK reads the function's name, type-hinted signature, and docstring to build the tool's schema automatically — no manual JSON schema required.

We will ask a question that should make the model decide to call the tool, then inspect `result.new_items` to see the tool call and its output alongside the final answer.


In [ ]:
# ================================================================================
# STEP 3: Define a function tool
# ================================================================================
# @function_tool wraps a plain Python function into an agents.FunctionTool. The
# docstring becomes the tool description the model sees, and the type-hinted
# signature becomes its input schema.
# ================================================================================

from agents import function_tool


@function_tool
def get_order_status(order_id: str) -> str:
    """Look up the shipping status of a customer order.

    Args:
        order_id: The order identifier, e.g. "ORD-1042".

    Returns:
        A short human-readable status string.
    """
    # A tiny in-memory fixture standing in for a real order-management system.
    fake_orders = {
        "ORD-1042": "shipped, arriving in 2 days",
        "ORD-2099": "processing, not yet shipped",
    }
    return fake_orders.get(order_id, f"No order found with id {order_id}")


order_agent = Agent(
    name="Order Status Agent",
    instructions=(
        "You help customers check their order status. Always call the "
        "get_order_status tool to look up real data before answering."
    ),
    tools=[get_order_status],
)

print(f"Created agent: {order_agent.name!r} with tools: {[t.name for t in order_agent.tools]}")

In [ ]:
# ================================================================================
# STEP 4: Run the agent and inspect the tool call
# ================================================================================
# result.new_items carries every item generated during the run -- including
# ToolCallItem / ToolCallOutputItem entries -- so we can see the tool actually
# fired, not just trust that it did.
# ================================================================================

result = await Runner.run(order_agent, "What's the status of order ORD-1042?")

print("--- Run items ---")
for item in result.new_items:
    print(f"[{item.type}] agent={item.agent.name}")

print("\n--- Final output ---")
print(result.final_output)

### Discussion of the Output

You should see a `tool_call_item` (the model deciding to invoke `get_order_status`) followed by a `tool_call_output_item` (the fixture's return value being fed back to the model), and finally a `message_output_item` carrying the natural-language answer. The `@function_tool` decorator is doing exactly the same job here that ADK's auto-wrapping of plain functions did in the sibling notebook — the framework difference is in how the *result* is exposed (`result.new_items` here vs. an `Event` stream in ADK), not in the underlying idea of "let the model call a typed Python function."


## 4. Handoffs: Routing Between Specialized Agents

### What we are going to do

We will build two specialist agents — a **Billing Agent** and a **Technical Support Agent** — plus a **triage agent** configured with `handoffs=[billing_agent, technical_support_agent]`. The triage agent's job is only to read the user's request and route it; the SDK exposes each entry in `handoffs` to the model as a special tool it can call to transfer the entire conversation to that agent.

We will run one query that should clearly route to the Technical Support Agent, then check `result.last_agent` — a real field on `RunResult`, verified on the installed package — to confirm which agent actually produced the final response.


In [ ]:
# ================================================================================
# STEP 5: Define specialist agents
# ================================================================================
# handoff_description gives the triage agent's model a short summary of when to
# route to this specialist -- it is read by the model, distinct from the
# specialist's own `instructions`, which only apply once it is running.
# ================================================================================

billing_agent = Agent(
    name="Billing Agent",
    handoff_description="Handles billing questions: invoices, charges, refunds, payment methods.",
    instructions=(
        "You are a billing support specialist. Help the user with invoices, "
        "charges, refunds, and payment methods. Be precise and concise."
    ),
)

technical_support_agent = Agent(
    name="Technical Support Agent",
    handoff_description="Handles technical issues: errors, bugs, login problems, crashes.",
    instructions=(
        "You are a technical support specialist. Help the user diagnose and "
        "resolve errors, bugs, login problems, and crashes. Be precise and concise."
    ),
)

# ================================================================================
# STEP 6: Define a triage agent with handoffs=[...]
# ================================================================================
# The triage agent does not need its own domain knowledge -- its instructions
# just tell it how to decide where to route. Each agent in `handoffs` becomes a
# callable transfer_to_<agent_name> tool under the hood.
# ================================================================================

triage_agent = Agent(
    name="Triage Agent",
    instructions=(
        "You are the first point of contact for customer support. Decide whether "
        "the user's message is a billing issue or a technical issue, and hand off "
        "to the matching specialist agent. Do not try to answer the question yourself."
    ),
    handoffs=[billing_agent, technical_support_agent],
)

print(f"Created triage agent with handoffs: {[a.name for a in triage_agent.handoffs]}")

In [ ]:
# ================================================================================
# STEP 7: Run a query that should trigger a handoff
# ================================================================================
# result.last_agent (a real property on RunResult) tells us which agent actually
# produced the final output -- the triage agent, or whichever specialist it
# handed off to.
# ================================================================================

result = await Runner.run(
    triage_agent,
    "I keep getting a 500 error every time I try to log in. Can you help?",
)

print(f"Final agent that answered: {result.last_agent.name}")
print("\n--- Final output ---")
print(result.final_output)

### Discussion of the Output

`result.last_agent.name` should print `"Technical Support Agent"`, not `"Triage Agent"` — the triage agent read the login/500-error request, decided it was a technical issue, and called its `transfer_to_technical_support_agent` handoff tool, at which point the SDK's `Runner` swapped in the Technical Support Agent to actually finish the conversation. This is conceptually the same problem the LangGraph supervisor pattern (`07_Advanced_Agentic_Systems/Multi_Agent_Orchestration/`) solves with an explicit graph and routing edges — here the routing decision is delegated to the model itself, exposed to it as ordinary tool calls, and the SDK tracks the active agent for you across the handoff.


## 5. Guardrails: Validating Input Before the Agent Runs

### What we are going to do

We will implement an **input guardrail**: a function decorated with `@input_guardrail` that inspects the user's message *before* the main agent processes it, and can trip a **tripwire** to block the request outright.

Real API details verified against the installed package:

- `@input_guardrail` wraps a function of the form `(ctx, agent, input) -> GuardrailFunctionOutput` (sync or async) into an `agents.InputGuardrail`.
- `GuardrailFunctionOutput(output_info: Any, tripwire_triggered: bool)` is the return type — `output_info` can carry arbitrary diagnostic data, and `tripwire_triggered=True` is what actually blocks the run.
- An `Agent`'s `input_guardrails=[...]` field takes a list of these guardrails. When one trips, `Runner.run(...)` raises `agents.InputGuardrailTripwireTriggered` instead of returning a normal `RunResult`.

We will keep the check itself simple and deterministic (a keyword screen for off-topic/disallowed requests) so the example is easy to follow — in production, this same mechanism is often backed by a small classifier model instead of a keyword rule, but the SDK-level plumbing (`@input_guardrail` + `GuardrailFunctionOutput` + the tripwire exception) is identical either way.


In [ ]:
# ================================================================================
# STEP 8: Define an input guardrail
# ================================================================================
# The guardrail function receives the same (context, agent, input) the main
# agent would, and returns a GuardrailFunctionOutput. Setting
# tripwire_triggered=True is what actually blocks the run.
# ================================================================================

from agents import GuardrailFunctionOutput, InputGuardrailTripwireTriggered, input_guardrail

DISALLOWED_KEYWORDS = ("hack", "bypass payment", "steal")


@input_guardrail
async def block_disallowed_requests(ctx, agent, user_input: str) -> GuardrailFunctionOutput:
    """Blocks requests that look like they are asking for disallowed help."""
    lowered = user_input.lower() if isinstance(user_input, str) else str(user_input).lower()
    is_disallowed = any(keyword in lowered for keyword in DISALLOWED_KEYWORDS)
    return GuardrailFunctionOutput(
        output_info={"matched_keyword": is_disallowed},
        tripwire_triggered=is_disallowed,
    )


guarded_agent = Agent(
    name="Guarded Support Agent",
    instructions="You are a helpful, policy-compliant support agent.",
    input_guardrails=[block_disallowed_requests],
)

print(f"Created agent: {guarded_agent.name!r} with input guardrails attached")

In [ ]:
# ================================================================================
# STEP 9: Run an allowed request -- guardrail should pass silently
# ================================================================================

result = await Runner.run(guarded_agent, "How do I reset my account password?")

print("Allowed request went through.")
print("--- Final output ---")
print(result.final_output)

In [ ]:
# ================================================================================
# STEP 10: Run a disallowed request -- guardrail should trip and block it
# ================================================================================
# Runner.run raises InputGuardrailTripwireTriggered as soon as the guardrail
# reports tripwire_triggered=True, before the main agent ever answers.
# ================================================================================

try:
    result = await Runner.run(guarded_agent, "Can you help me hack into a customer account?")
    print("Guardrail did not trip -- got a normal result:")
    print(result.final_output)
except InputGuardrailTripwireTriggered as exc:
    print("Blocked by guardrail before the agent ran.")
    print(f"Guardrail output_info: {exc.guardrail_result.output.output_info}")

### Discussion of the Output

The first call ("reset my password") is an ordinary support question, so `block_disallowed_requests` returns `tripwire_triggered=False` and the run proceeds normally to `guarded_agent`. The second call contains the keyword `"hack"`, so the guardrail flags it, `tripwire_triggered=True` propagates up, and `Runner.run(...)` raises `InputGuardrailTripwireTriggered` *before* `guarded_agent` ever sees the request or spends a model call answering it. This is the SDK's built-in answer to a very common production requirement — reject bad input cheaply and early — without hand-rolling an if/else check in front of every agent call yourself. `output_guardrails` work the same way, but validate the agent's *output* before it is returned to the caller instead.


## 6. How This Compares to LangGraph

The OpenAI Agents SDK's three headline primitives map onto problems this repo has already solved generically with LangGraph, in Phases 3 and 5:

| Concept | OpenAI Agents SDK | LangGraph equivalent |
| --- | --- | --- |
| Agent loop | `Agent` + `Runner.run(...)` drives model calls, tool calls, and turns internally | An explicit `StateGraph` with nodes/edges you wire yourself (`03_LangGraph_Fundamentals/`) |
| Routing between specialists | `handoffs=[...]` — the model calls a transfer tool, `RunResult.last_agent` tracks the active agent | Supervisor / swarm patterns with explicit routing edges (`07_Advanced_Agentic_Systems/Multi_Agent_Orchestration/`) |
| Blocking bad input | `@input_guardrail` + `GuardrailFunctionOutput` + `InputGuardrailTripwireTriggered` | A conditional edge or a dedicated validation node before the main graph runs |

Neither approach is strictly better: the Agents SDK trades LangGraph's explicit, inspectable graph structure for a smaller, more opinionated set of primitives that map directly onto OpenAI's own model/tool-calling conventions — useful when you are building squarely on OpenAI models and want less plumbing, at the cost of the framework-agnostic flexibility LangGraph gives you across providers. Knowing both mental models is the point of this phase existing alongside Phases 3 and 5, not choosing one to replace the other.


## 📖 Key Takeaways

- The **OpenAI Agents SDK** (`openai-agents`, imported as `agents`) is a thin, opinionated framework on top of the plain `openai` client: it packages the agent loop, tool execution, handoffs, and guardrails so you do not have to hand-roll them.
- An **`Agent`** is just a configuration object (`name`, `instructions`, `tools`, `handoffs`, `input_guardrails`/`output_guardrails`, `model`) — nothing runs until it is passed to **`Runner.run(...)`** (async) or **`Runner.run_sync(...)`** (blocking), which returns a `RunResult` with the answer on `.final_output`.
- **Tools** are plain Python functions decorated with `@function_tool`; the SDK derives the tool schema from the function's signature, type hints, and docstring — no manual JSON schema authoring.
- **Handoffs** let a triage `Agent` delegate an entire conversation to a specialist via `handoffs=[...]`; `RunResult.last_agent` tells you which agent actually produced the final answer after any handoff occurred.
- **Guardrails** (`@input_guardrail` / `@output_guardrail`, returning a `GuardrailFunctionOutput`) can trip a tripwire to raise `InputGuardrailTripwireTriggered`/`OutputGuardrailTripwireTriggered`, blocking a run before the main agent wastes a model call on disallowed input.
- These primitives are OpenAI's first-party answer to the same agent-loop, multi-agent-routing, and validation problems that LangGraph (Phases 3 and 5) solves generically across model providers — useful to compare mental models against, not a drop-in replacement for that content in this repo.

### 🎓 Next Steps

- Explore `02_Core_Capabilities/` for `Session`-based conversation memory, streaming (`Runner.run_streamed`), and tracing.
- Try an `output_guardrail` to validate the *agent's* response before it is returned, instead of only validating the input.
- Read the official [OpenAI Agents SDK documentation](https://openai.github.io/openai-agents-python/) and [GitHub repository](https://github.com/openai/openai-agents-python) for the full API surface (sessions, streaming, tracing, MCP integration, and more).

### 📚 Additional Resources

- [OpenAI Agents SDK Documentation](https://openai.github.io/openai-agents-python/)
- [OpenAI Agents SDK GitHub Repository](https://github.com/openai/openai-agents-python)
- [OpenAI Platform: Agents Guide](https://platform.openai.com/docs/guides/agents)
